# Supp Table 4a: ECM neighborhoods in TIL zones and TIL groups

**Produces** two Supp Table 4a files:
- `ecm_neighborhood_TIL_group_by_zone_pseudobulk`: per TIL zone and ECM neighborhood, the mean
  per-section fraction of the zone's spots in that neighborhood in TIL-high and TIL-low sections,
  difference, log2 ratio, Mann–Whitney p and BH FDR within each zone.
- `ecm_neighborhood_niche_matrisome_summary`: per ECM neighborhood, the number of spots, the
  fraction of its spots in each TIL zone and in TIL-high / TIL-low sections, and the mean
  log-normalised expression of 96 matrisome genes over its spots.

**Origin**: Spencer Watson, EFS `Analysis_Repo/Notebooks/TIL_Niche_Comparisons_v2_clean.ipynb`:
cells 70 (ECM labels onto the spots) and 71 (pseudobulk table, written 2025-12-12 to
`results/visium/ecm/`); the matrisome gene list is that of cell 72. No code was found for the
matrisome summary (written 2025-12-10 to the same folder; repo history, EFS notebooks searched);
it is reconstructed here from its columns (spot-level fractions and means).

**What the published tables were computed on** (`AS_PUBLISHED = True`, default):
- the ECM labels of `Analysis_Repo/Notebooks/ecm_neighborhood_annotations.csv`, an earlier ECM
  clustering (240 sections), **not** the v4 neighborhoods used everywhere else in the paper
  (`ecm_neighborhood_annotations_v4.csv`; only 13 % of spots have the same label number);
- 229 Visium sections: the 228 of the paper plus CR_G_335a_vis, excluded from the cohort on
  2025-12-12 (`PATIENT_EXCLUSION`, clinical label missing). Its TIL zones are recomputed here with
  the rule of `fig5_0_TIL_zones.ipynb`; with it the upper-quartile cut-off is 0.0300 and the
  groups are 58 TIL-high / 171 TIL-low sections (paper: 57 / 171).

`AS_PUBLISHED = False` computes the same tables on the paper's cohort and labels (228 sections,
57 / 171, v4 neighborhoods); both versions are written, the panel files follow `AS_PUBLISHED`.

**Compared with the published tables**: see the last cells.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests

from gbmcollaboration import AllDataCenters, CRDataCenter
from gbmcollaboration.paper import annotation, config, output, til, visium

In [ ]:
OUT = output.setup("supptables_4a_ecm_til_zone_tables")
PUBLISHED_TABLES = config.EFS / "supplementary_tables"
AS_PUBLISHED = True
ECM_PATH_PUBLISHED = config.EFS / "Analysis_Repo/Notebooks/ecm_neighborhood_annotations.csv"
EXTRA_SECTION = "CR_G_335a_vis"

# TIL zone rule of fig5_0_TIL_zones.ipynb
TIL_LEVEL_COL = "celltype_level2_curated"
TIL_GROUPS = ["T_NK"]
TIL_THRESHOLD = 2
K = 6

SAMPLE_COL = "sample"
REGION_COL = "TIL_zone"
GROUP_COL  = "TIL_group"
ECM_COL    = "ECM_neighborhood"

ZONES      = ["TIL+", "BZ1", "BZ2","TIL-"]
GROUPS     = ["TIL_high", "TIL_low"]

# Matrisome / ECM genes (brain-focused list)
ecm_genes_brain = [
    # Original list
    "COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2",
    "COL5A1", "COL6A1", "COL6A2", "COL6A3", "COL7A1",
    "COL8A1", "COL10A1", "COL11A1", "COL12A1",
    "COL14A1", "COL15A1", "COL16A1",
    "FN1", "FBN1", "FBN2", "TNC", "SPARC", "POSTN",
    "LAMA1", "LAMA2", "LAMA3", "LAMB1", "LAMB2", "LAMC1",
    "THBS1", "THBS2", "THBS4", "VTN", "NID1", "NID2",
    "VCAN", "DCN", "BGN", "LUM", "HSPG2", "AGRN", "GPC1", "GPC3",
    "MMP1", "MMP2", "MMP9", "MMP11", "MMP14",
    "ADAMTS1", "ADAMTS4", "ADAMTS5", "TIMP1", "TIMP2",
    "LOX", "LOXL1", "SERPINE1", "PLAU", "PLAT",
    "CSPG4", "BCAN", "NCAN", "HAPLN1", "HAPLN2", "PRELP", "FMOD",
    "COL18A1", "LOXL2", "LOXL3",
    "HPSE", "SDC1", "GPC2", "CSPG5", "PTPRZ1",

    # Added structural ECM
    "COL5A2", "COL5A3",
    "LAMA4", "LAMA5",
    "ACAN",
    "TNR",
    "HAPLN3", "HAPLN4",
    "FBLN1", "FBLN2", "FBLN5",
    "SDC4",

    # Added matricellular / ECM-associated
    "SPP1",
    "IGFBP2", "IGFBP5", "SRGN",

    # Added ECM remodelers
    "MMP3", "MMP7", "MMP12",
    "ADAMTS8", "ADAM8",
    "TIMP3",
    "LOXL4",
]

In [ ]:
def section_til_zones(section):
    """TIL+ / BZ1 / BZ2 / TIL- of one section, with the rule of fig5_0_TIL_zones.ipynb."""
    loaded = CRDataCenter.load_spatial(exclude_low_quality_visium=True, exclude_idh1_mut=True,
                                       exclude_patient_ids=False, exclude_relapse=True,
                                       list_sample_id=[section], normalization="log_norm_count", radius=1,
                                       deconvolution_slot_name="deconv_cohort_level_4",
                                       load_gex=False, load_deconvolution=True)
    obs = loaded.celltype_df[["imagecol", "imagerow"]].copy()
    obs.index = [f"{s}_{b}" for s, b in obs.index]
    c2l = CRDataCenter.load_cell2location([section])
    c2l.index = [f"{s}_{b}" for s, b in c2l.index]
    key = annotation.celltype_key(efs_copy=True)
    til_cols = [f"{visium.C2L_PREFIX}{ct}" for ct in key.index[key[TIL_LEVEL_COL].isin(TIL_GROUPS)]]
    tilp = (c2l.reindex(obs.index).reindex(columns=til_cols).sum(axis=1) >= TIL_THRESHOLD).to_numpy()
    xy = obs[["imagecol", "imagerow"]].to_numpy(dtype=float)
    _, nbrs = NearestNeighbors(n_neighbors=min(K + 1, len(xy)), metric="euclidean").fit(xy).kneighbors(xy)
    bz1 = np.zeros(len(xy), dtype=bool)
    for i in np.where(tilp)[0]:
        bz1[nbrs[i][nbrs[i] != i]] = True
    bz1 &= ~tilp
    bz2 = np.zeros(len(xy), dtype=bool)
    for i in np.where(bz1)[0]:
        bz2[nbrs[i][nbrs[i] != i]] = True
    bz2 &= ~(tilp | bz1)
    zone = np.select([tilp, bz1, bz2], ["TIL+", "BZ1", "BZ2"], default="TIL-")
    return pd.DataFrame({"sample": section, "TIL_positive": tilp, "TIL_zone": zone}, index=obs.index)


def til_group_by_section(zones):
    """TIL_high / TIL_low per section: upper quartile of the TIL+ spot fraction."""
    frac = zones.groupby("sample")["TIL_positive"].mean()
    return pd.Series(np.where(frac >= frac.quantile(0.75), "TIL_high", "TIL_low"), index=frac.index)


def ecm_pseudobulk_table(obs):
    """Cell 71: per-section fractions of each ECM neighborhood within each zone, high vs low."""
    obs = obs[[SAMPLE_COL, REGION_COL, GROUP_COL, ECM_COL]].copy()
    obs = obs[obs[REGION_COL].isin(ZONES)].copy()
    obs = obs.dropna(subset=[SAMPLE_COL, REGION_COL, GROUP_COL, ECM_COL])

    # groupby(sample, zone) then compute fraction of each ECM neighborhood
    ecm_frac = (
        obs
        .groupby([SAMPLE_COL, REGION_COL])[ECM_COL]
        .value_counts(normalize=True)
        .rename("frac")
        .reset_index()
    )
    ecm_frac = ecm_frac.rename(columns={REGION_COL: "zone"})

    # Attach TIL_group per sample
    sample_group = obs[[SAMPLE_COL, GROUP_COL]].drop_duplicates()
    ecm_frac = ecm_frac.merge(sample_group, on=SAMPLE_COL, how="left")

    rows = []
    for zone in ZONES:
        df_zone = ecm_frac[ecm_frac["zone"] == zone].copy()
        if df_zone.empty:
            print(f"No data for zone {zone}, skipping.")
            continue

        for ecm_id in sorted(df_zone[ECM_COL].unique()):
            df_ecm = df_zone[df_zone[ECM_COL] == ecm_id]

            vals_high = df_ecm.loc[df_ecm[GROUP_COL] == "TIL_high", "frac"].values
            vals_low  = df_ecm.loc[df_ecm[GROUP_COL] == "TIL_low" , "frac"].values

            # Mean fractions
            mean_high = np.nanmean(vals_high) if len(vals_high) > 0 else np.nan
            mean_low  = np.nanmean(vals_low)  if len(vals_low)  > 0 else np.nan
            delta     = mean_high - mean_low

            # Log2 fold change with epsilon to avoid division by zero
            eps = 1e-6
            log2fc = np.log2((mean_high + eps) / (mean_low + eps))

            # Mann-Whitney U test if both groups have at least 3 samples
            if len(vals_high) >= 3 and len(vals_low) >= 3:
                try:
                    _, pval = mannwhitneyu(vals_high, vals_low, alternative="two-sided")
                except ValueError:
                    pval = np.nan
            else:
                pval = np.nan

            rows.append(
                {
                    "zone": zone,
                    ECM_COL: ecm_id,
                    "mean_frac_TIL_high": mean_high,
                    "mean_frac_TIL_low": mean_low,
                    "delta_high_minus_low": delta,
                    "log2fc_high_over_low": log2fc,
                    "pval": pval,
                    "n_samples_high": int((df_ecm[GROUP_COL] == "TIL_high").sum()),
                    "n_samples_low":  int((df_ecm[GROUP_COL] == "TIL_low").sum()),
                }
            )

    df_ecm_neigh = pd.DataFrame(rows)

    # FDR correction per zone
    df_ecm_neigh["fdr"] = np.nan
    for zone in ZONES:
        mask = (df_ecm_neigh["zone"] == zone) & df_ecm_neigh["pval"].notna()
        if mask.sum() > 0:
            _, fdrs, _, _ = multipletests(df_ecm_neigh.loc[mask, "pval"], method="fdr_bh")
            df_ecm_neigh.loc[mask, "fdr"] = fdrs
    return df_ecm_neigh


def matrisome_summary(obs, expr):
    """Per ECM neighborhood: spots, fraction in each zone and TIL group, mean gene expression."""
    ecm = obs[ECM_COL]
    out = pd.DataFrame({"n_spots": ecm.value_counts().sort_index()})
    out.index.name = ECM_COL
    zone_frac = pd.crosstab(ecm, obs[REGION_COL], normalize="index")
    for zone in ZONES:
        out[f"frac_zone_{zone}"] = zone_frac[zone]
    group_frac = pd.crosstab(ecm, obs[GROUP_COL], normalize="index")
    for group in GROUPS:
        out[f"frac_TILgroup_{group}"] = group_frac[group]
    means = expr.astype(np.float64).groupby(ecm.reindex(expr.index)).mean()
    return out.join(means.add_prefix("mean_")).reset_index()


def compare(name, new):
    pub = pd.read_csv(PUBLISHED_TABLES / f"5a_{name}.csv")
    print(f"{name}: shape published {pub.shape}, this run {new.shape}; same columns {list(pub.columns) == list(new.columns)}")
    num = pub.select_dtypes("number").columns
    other = pub.columns.difference(num)
    if len(other):
        print("  text columns identical:", (pub[other].astype(str) == new[other].astype(str)).all().all())
    diff = (pub[num] - new[num].to_numpy()).abs().max()
    print(f"  max abs difference: {diff.max():.3g} ({diff.idxmax()})")

## Spots: TIL zones, TIL groups, ECM labels

In [ ]:
zones = til.til_zones()[["sample", "TIL_positive", "TIL_zone"]]
ecm_published = pd.read_csv(ECM_PATH_PUBLISHED, index_col=0)
ecm_v4 = pd.read_csv(config.ECM_ANNOTATIONS_CSV, index_col="spot_id")

zones_published = pd.concat([zones, section_til_zones(EXTRA_SECTION)])
print(f"{EXTRA_SECTION}: {zones_published.query('sample == @EXTRA_SECTION')['TIL_zone'].value_counts().to_dict()}")

variants = {}
for name, z, ecm in [("as_published", zones_published, ecm_published), ("paper_cohort_v4", zones, ecm_v4)]:
    groups = til_group_by_section(z)
    obs = z.copy()
    obs[REGION_COL] = pd.Categorical(obs[REGION_COL], categories=["TIL-", "BZ2", "BZ1", "TIL+"], ordered=True)
    obs[GROUP_COL] = pd.Categorical(obs[SAMPLE_COL].map(groups), categories=GROUPS)
    obs = obs.join(ecm[[ECM_COL]], how="inner")
    obs[ECM_COL] = obs[ECM_COL].astype(int)
    variants[name] = obs
    print(f"{name}: {obs[SAMPLE_COL].nunique()} sections, {len(obs):,} spots; "
          f"sections per group {groups.value_counts().to_dict()}")

## Matrisome expression (log-normalised, per spot)

In [ ]:
sections = sorted(set().union(*[v[SAMPLE_COL].unique() for v in variants.values()]))
ad = AllDataCenters.load_spatial_adata(exclude_patient_ids=False, genes=ecm_genes_brain, list_sample_id=sections)
expr = pd.DataFrame(ad.layers["log_norm_count"], index=ad.obs_names, columns=ad.var_names)[ecm_genes_brain]
del ad
print(f"expression: {expr.shape}")

## Tables

In [ ]:
tables = {}
for name, obs in variants.items():
    tables[name] = {
        "ecm_neighborhood_TIL_group_by_zone_pseudobulk": ecm_pseudobulk_table(obs),
        "ecm_neighborhood_niche_matrisome_summary": matrisome_summary(obs, expr.reindex(obs.index)),
    }
    for stem, df in tables[name].items():
        output.save_table(OUT, f"{stem}_{name}", df, index=False)

panel_variant = "as_published" if AS_PUBLISHED else "paper_cohort_v4"
for stem, df in tables[panel_variant].items():
    output.save_table(OUT, stem, df, panel=f"Supp Table 4a {stem}", index=False)
tables[panel_variant]["ecm_neighborhood_TIL_group_by_zone_pseudobulk"].head(10)

## Comparison with the published tables

In [ ]:
for name in variants:
    print(f"--- {name}")
    for stem, df in tables[name].items():
        compare(stem, df)
for stem in tables[panel_variant]:
    same = (OUT / f"{stem}.csv").read_bytes() == (PUBLISHED_TABLES / f"5a_{stem}.csv").read_bytes()
    print(f"{stem} ({panel_variant}): byte-identical to the published file: {same}")

In [ ]:
print(output.versions(["numpy", "pandas", "scipy", "statsmodels", "scikit-learn"]).to_string(index=False))